<a href="https://colab.research.google.com/github/shanusushmita/CS-6140-Machine-Learning-Course-/blob/main/Simple_Text_Classification_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Example image](https://upload.wikimedia.org/wikipedia/commons/0/02/Northeastern_Wordmark.svg)

## Text Classification for Sentiments

Copyright: Prof. Shanu Sushmita

In [2]:
"""
Simple TF-IDF Sentiment Analysis Example
"""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Simple training dataset - movie reviews
documents = [
    # Positive reviews (label = 1)
    "This movie is fantastic and amazing",
    "I loved this film it was wonderful",
    "Great movie excellent acting",
    "Best film I have ever seen",
    "Absolutely brilliant and entertaining",
    "Wonderful story and great characters",
    "This is an excellent movie",
    "I really enjoyed this film",

    # Negative reviews (label = 0)
    "This movie is terrible and boring",
    "I hated this film it was awful",
    "Worst movie bad acting",
    "Terrible film I never want to see again",
    "Absolutely horrible and dull",
    "Waste of time and money",
    "This is a terrible movie",
    "I really disliked this film"
]

labels = [1, 1, 1, 1, 1, 1, 1, 1,  # Positive
          0, 0, 0, 0, 0, 0, 0, 0]  # Negative

# Test documents
test_documents = [
    "This film is excellent and entertaining",  # Should be positive
    "Boring and terrible waste of time",        # Should be negative
    "Great acting wonderful story",             # Should be positive
    "Awful movie horrible experience"           # Should be negative
]

print("="*70)
print("STEP 1: CREATING TF-IDF FEATURES")
print("="*70)

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=20  # Keep top 20 words for simplicity
)

# Fit and transform training documents
X_train_tfidf = vectorizer.fit_transform(documents)

print(f"\nVocabulary (top words): {vectorizer.get_feature_names_out()}\n")

# Show TF-IDF scores for first document
print("Example: TF-IDF scores for first document:")
print(f"Document: '{documents[0]}'")
print(f"Label: {'Positive' if labels[0] == 1 else 'Negative'}\n")

# Get feature names and scores
feature_names = vectorizer.get_feature_names_out()
first_doc_scores = X_train_tfidf[0].toarray()[0]

# Show non-zero scores
print("Word          | TF-IDF Score")
print("-" * 35)
for word, score in zip(feature_names, first_doc_scores):
    if score > 0:
        print(f"{word:13} | {score:.4f}")

print("\n" + "="*70)
print("STEP 2: TRAINING CLASSIFIERS")
print("="*70)

# Train Naive Bayes classifier
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_tfidf, labels)
print("\n✓ Naive Bayes classifier trained")

# Train Logistic Regression classifier
lr_classifier = LogisticRegression(max_iter=1000)
lr_classifier.fit(X_train_tfidf, labels)
print("✓ Logistic Regression classifier trained")

print("\n" + "="*70)
print("STEP 3: MAKING PREDICTIONS ON TEST DATA")
print("="*70)

# Transform test documents
X_test_tfidf = vectorizer.transform(test_documents)

# Make predictions with both classifiers
nb_predictions = nb_classifier.predict(X_test_tfidf)
lr_predictions = lr_classifier.predict(X_test_tfidf)

# Get prediction probabilities
nb_proba = nb_classifier.predict_proba(X_test_tfidf)
lr_proba = lr_classifier.predict_proba(X_test_tfidf)

print("\nTest Results:")
print("-" * 70)
for i, doc in enumerate(test_documents):
    print(f"\nDocument: '{doc}'")
    print(f"  Naive Bayes:        {['Negative', 'Positive'][nb_predictions[i]]} (confidence: {nb_proba[i][nb_predictions[i]]:.2%})")
    print(f"  Logistic Regression: {['Negative', 'Positive'][lr_predictions[i]]} (confidence: {lr_proba[i][lr_predictions[i]]:.2%})")

print("\n" + "="*70)
print("STEP 4: SHOWING MOST IMPORTANT WORDS")
print("="*70)

# For Logistic Regression, show most important words for each class
lr_coef = lr_classifier.coef_[0]

print("\nMost important words for POSITIVE sentiment:")
positive_words = sorted(zip(feature_names, lr_coef), key=lambda x: x[1], reverse=True)[:5]
for word, score in positive_words:
    print(f"  {word:15} (weight: {score:+.3f})")

print("\nMost important words for NEGATIVE sentiment:")
negative_words = sorted(zip(feature_names, lr_coef), key=lambda x: x[1])[:5]
for word, score in negative_words:
    print(f"  {word:15} (weight: {score:+.3f})")

print("\n" + "="*70)
print("KEY CONCEPTS:")
print("="*70)
print("""
1. TF-IDF converts text into numerical features
   - TF (Term Frequency): How often a word appears in a document
   - IDF (Inverse Document Frequency): How rare/common a word is across all documents
   - Higher TF-IDF = more important word for that specific document

2. Each document becomes a vector of TF-IDF scores
   - Dimensions = vocabulary size (20 words in this example)
   - Most values are zero (sparse matrix)

3. Classifiers learn patterns in these vectors
   - Positive reviews: high scores for "excellent", "great", "loved"
   - Negative reviews: high scores for "terrible", "awful", "worst"

4. New documents are transformed the same way and classified
   - Must use the SAME vocabulary learned during training
""")

print("\n" + "="*70)
print("INTERACTIVE SECTION: TRY YOUR OWN REVIEW!")
print("="*70)

print("\nEnter your own movie review (or press Enter to skip):")
user_input = input("> ")

if user_input.strip():
    user_tfidf = vectorizer.transform([user_input])
    user_nb_pred = nb_classifier.predict(user_tfidf)[0]
    user_lr_pred = lr_classifier.predict(user_tfidf)[0]
    user_nb_conf = nb_classifier.predict_proba(user_tfidf)[0][user_nb_pred]
    user_lr_conf = lr_classifier.predict_proba(user_tfidf)[0][user_lr_pred]

    print(f"\nYour review: '{user_input}'")
    print(f"  Naive Bayes:        {['Negative', 'Positive'][user_nb_pred]} (confidence: {user_nb_conf:.2%})")
    print(f"  Logistic Regression: {['Negative', 'Positive'][user_lr_pred]} (confidence: {user_lr_conf:.2%})")

    # Show TF-IDF scores for user's input
    print("\nTF-IDF scores in your review:")
    user_scores = user_tfidf.toarray()[0]
    for word, score in zip(feature_names, user_scores):
        if score > 0:
            print(f"  {word:15} | {score:.4f}")
else:
    print("Skipping interactive section.")

print("\n" + "="*70)
print("END OF DEMONSTRATION")
print("="*70)

STEP 1: CREATING TF-IDF FEATURES

Vocabulary (top words): ['absolutely' 'acting' 'amazing' 'awful' 'bad' 'best' 'boring' 'brilliant'
 'characters' 'dull' 'enjoyed' 'entertaining' 'excellent' 'fantastic'
 'film' 'great' 'movie' 'really' 'terrible' 'wonderful']

Example: TF-IDF scores for first document:
Document: 'This movie is fantastic and amazing'
Label: Positive

Word          | TF-IDF Score
-----------------------------------
amazing       | 0.6508
fantastic     | 0.6508
movie         | 0.3911

STEP 2: TRAINING CLASSIFIERS

✓ Naive Bayes classifier trained
✓ Logistic Regression classifier trained

STEP 3: MAKING PREDICTIONS ON TEST DATA

Test Results:
----------------------------------------------------------------------

Document: 'This film is excellent and entertaining'
  Naive Bayes:        Positive (confidence: 66.06%)
  Logistic Regression: Positive (confidence: 61.39%)

Document: 'Boring and terrible waste of time'
  Naive Bayes:        Negative (confidence: 77.61%)
  Logist